# 第 9 章 图神经网络

很多现实世界的数据天然是**图**（graph）结构的：社交网络、分子结构、知识图谱、引用网络、推荐系统……图数据具有不规则的拓扑：节点数量可变、邻居数量不定、节点之间通过边连接。前面学过的 MLP、CNN、RNN 都假设输入是**规则的**张量（向量、网格、序列），无法直接处理这种不规则结构。

**图神经网络**（Graph Neural Network，GNN）通过"在图上做消息传递"来学习节点/边/图的表示。本章实现几种最具代表性的 GNN 模型：

- **GCN**（Graph Convolutional Network，Kipf & Welling, 2017）：图卷积；
- **GraphSAGE**（Hamilton et al., 2017）：邻居采样 + 聚合；
- **GAT**（Graph Attention Network，Veličković et al., 2018）：图注意力；
- **GIN**（Graph Isomorphism Network，Xu et al., 2019）：理论表达力更强。

本章不依赖 `torch_geometric` 等第三方 GNN 库，全部用纯 PyTorch 从零实现，重点是理解消息传递的机制。

> **拓展阅读**：实际项目中推荐使用 [PyTorch Geometric (PyG)](https://pytorch-geometric.readthedocs.io/)，它提供了完整的数据集、采样器和 GNN 模型库。本章末尾给出一个 PyG 风格的实现对比。


## 1. 图数据的基本表示

一个图 $G = (V, E)$ 由节点集 $V$ 和边集 $E$ 组成。常用表示：

- **邻接矩阵** $\boldsymbol{A} \in \mathbb{R}^{N\times N}$：$A_{ij} = 1$ 当节点 $i, j$ 有边相连。
- **节点特征矩阵** $\boldsymbol{X} \in \mathbb{R}^{N\times D}$：每个节点一个 $D$ 维特征向量。
- **度矩阵** $\boldsymbol{D}$：对角矩阵，$D_{ii} = \sum_j A_{ij}$ 是节点 $i$ 的度数。

下面用一个小例子说明：

> 经典的 **Zachary's Karate Club** 数据集——一个 34 节点的社交网络，描述一个空手道俱乐部中成员的友谊关系。后来俱乐部分裂成两派（围绕教练 Mr. Hi 和管理员 Officer），我们用 GNN 学习每个成员属于哪一派。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(0)


# ============== 构造 Karate Club 图（34 节点） ==============
# 边列表来自 Zachary 1977 原始数据
edges_raw = [
    (0,1),(0,2),(0,3),(0,4),(0,5),(0,6),(0,7),(0,8),(0,10),(0,11),
    (0,12),(0,13),(0,17),(0,19),(0,21),(0,31),
    (1,2),(1,3),(1,7),(1,13),(1,17),(1,19),(1,21),(1,30),
    (2,3),(2,7),(2,8),(2,9),(2,13),(2,27),(2,28),(2,32),
    (3,7),(3,12),(3,13),
    (4,6),(4,10),
    (5,6),(5,10),(5,16),
    (6,16),
    (8,30),(8,32),(8,33),
    (9,33),
    (13,33),
    (14,32),(14,33),
    (15,32),(15,33),
    (18,32),(18,33),
    (19,33),
    (20,32),(20,33),
    (22,32),(22,33),
    (23,25),(23,27),(23,29),(23,32),(23,33),
    (24,25),(24,27),(24,31),
    (25,31),
    (26,29),(26,33),
    (27,33),
    (28,31),(28,33),
    (29,32),(29,33),
    (30,32),(30,33),
    (31,32),(31,33),
    (32,33),
]
N = 34

# 邻接矩阵
A = torch.zeros(N, N)
for i, j in edges_raw:
    A[i, j] = 1
    A[j, i] = 1                                 # 无向图

# 度矩阵
D = A.sum(dim=1)                                 # 每个节点的度

# 节点标签：0=Mr. Hi 派系, 1=Officer 派系
y = torch.tensor([0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0,0,1,0,1,0,1,1,1,1,1,1,1,1,1,1,1,1])

# 节点特征：先用 one-hot（特征数=节点数），相当于"我就是 i"，让模型自己学
X = torch.eye(N)

print(f'nodes: {N}    edges: {len(edges_raw)}    feature dim: {X.shape[1]}')
print(f'class 0 (Mr. Hi):  {(y == 0).sum().item()} nodes')
print(f'class 1 (Officer): {(y == 1).sum().item()} nodes')
print(f'avg degree: {D.mean().item():.2f}')


## 2. 消息传递机制：GNN 的核心抽象

几乎所有的 GNN 都遵循同一个**消息传递**（message passing）框架：每层 GNN 做三件事：

1. **消息计算（message）**：对每条边 $(i, j)$，从源节点 $j$ 算出要传给目标节点 $i$ 的消息 $\boldsymbol{m}_{j\to i}$；
2. **邻居聚合（aggregate）**：节点 $i$ 把来自所有邻居 $\mathcal{N}(i)$ 的消息聚合起来，比如求和或求平均；
3. **节点更新（update）**：用聚合的消息和节点自身的旧表示 $\boldsymbol{h}_i$，算出新表示 $\boldsymbol{h}'_i$。

用一个抽象公式：

$$ \boldsymbol{h}_i' = \mathrm{Update}\Big(\boldsymbol{h}_i,\ \mathrm{Aggregate}\big\{ \mathrm{Message}(\boldsymbol{h}_j,\ \boldsymbol{h}_i,\ \boldsymbol{e}_{ij})\ :\ j \in \mathcal{N}(i) \big\}\Big). $$

不同的 GNN 模型本质上就是给这三个函数选择不同的形式。下面我们用矩阵形式高效实现。


## 3. GCN：图卷积网络

Kipf & Welling 提出的 GCN 用一个紧凑的矩阵公式实现消息传递：

$$ \boldsymbol{H}^{(l+1)} = \sigma\Big(\tilde{\boldsymbol{D}}^{-\frac{1}{2}}\ \tilde{\boldsymbol{A}}\ \tilde{\boldsymbol{D}}^{-\frac{1}{2}}\ \boldsymbol{H}^{(l)}\ \boldsymbol{W}^{(l)}\Big), $$

其中：

- $\tilde{\boldsymbol{A}} = \boldsymbol{A} + \boldsymbol{I}$：加自环（让每个节点也"传消息给自己"）；
- $\tilde{\boldsymbol{D}}$：$\tilde{\boldsymbol{A}}$ 的度矩阵；
- $\tilde{\boldsymbol{D}}^{-\frac{1}{2}} \tilde{\boldsymbol{A}} \tilde{\boldsymbol{D}}^{-\frac{1}{2}}$：对称归一化的邻接矩阵，防止度数大的节点主导聚合；
- $\boldsymbol{W}^{(l)}$：可学习的权重矩阵；
- $\sigma$：非线性激活，通常 ReLU。

> 直觉：把每个节点 $i$ 的表示更新为"邻居（含自己）的表示按度数归一化后的加权和，再做线性变换 + 非线性"。


In [ ]:
def normalize_adj(A):
    """GCN 的对称归一化：A_hat = D^{-1/2} (A + I) D^{-1/2}."""
    N = A.shape[0]
    A_tilde = A + torch.eye(N)
    D_tilde = A_tilde.sum(dim=1)
    D_inv_sqrt = D_tilde.pow(-0.5)
    D_inv_sqrt[torch.isinf(D_inv_sqrt)] = 0.0
    D_mat = torch.diag(D_inv_sqrt)
    return D_mat @ A_tilde @ D_mat


class GCNLayer(nn.Module):
    """一层 GCN：H' = sigma(A_hat H W)."""

    def __init__(self, in_dim, out_dim, bias=True):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=bias)

    def forward(self, H, A_hat):
        # H: [N, in_dim], A_hat: [N, N]
        return A_hat @ self.lin(H)


class GCN(nn.Module):
    """两层 GCN：[N, in] -> [N, hidden] -> [N, out]."""

    def __init__(self, in_dim, hidden, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, hidden)
        self.conv2 = GCNLayer(hidden, out_dim)
        self.dropout = dropout

    def forward(self, H, A_hat):
        H = F.relu(self.conv1(H, A_hat))
        H = F.dropout(H, p=self.dropout, training=self.training)
        return self.conv2(H, A_hat)


# 训练 GCN 做 Karate Club 节点分类
A_hat = normalize_adj(A)

torch.manual_seed(0)
model = GCN(in_dim=N, hidden=16, out_dim=2, dropout=0.3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05, weight_decay=5e-4)

# 半监督设定：只用 2 个节点的标签（每类 1 个）做训练
train_mask = torch.zeros(N, dtype=torch.bool)
train_mask[0]  = True       # Mr. Hi 派
train_mask[33] = True       # Officer 派

for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    logits = model(X, A_hat)
    loss = F.cross_entropy(logits[train_mask], y[train_mask])
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        with torch.no_grad():
            model.eval()
            pred = model(X, A_hat).argmax(dim=1)
            acc = (pred == y).float().mean().item()
        print(f'epoch {epoch+1:3d}  loss {loss.item():.4f}  full acc {acc:.4f}')


## 4. GraphSAGE：邻居采样 + 聚合

GCN 的局限：每次更新都要算整个邻接矩阵 $\hat{\boldsymbol{A}}$，对**大图**（百万节点）显存放不下。

**GraphSAGE** 解决这个问题：把更新公式直接写成针对单个节点 $i$ 的形式，并支持**邻居采样**——每次只从 $\mathcal{N}(i)$ 里采 $K$ 个邻居参与聚合，能 scale 到任意大图。

$$ \boldsymbol{h}_i^{(l+1)} = \sigma\Big(\boldsymbol{W}^{(l)} \cdot \mathrm{CONCAT}\big(\boldsymbol{h}_i^{(l)},\ \mathrm{AGG}\{\boldsymbol{h}_j^{(l)} : j \in \mathcal{N}(i)\}\big)\Big), $$

其中 AGG 可以是 mean / sum / LSTM / max-pool。下面实现 mean aggregator 版本，并演示一层小批量训练时的邻居采样。


In [ ]:
class SAGELayer(nn.Module):
    """GraphSAGE 一层 (mean aggregator)：concat[h_i, mean(N(i))] -> linear."""

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(2 * in_dim, out_dim)

    def forward(self, H, A):
        # H: [N, in_dim]; A: [N, N]（0/1 邻接矩阵，不需要自环）
        # 邻居均值
        deg = A.sum(dim=1, keepdim=True).clamp(min=1)
        nbr_mean = (A @ H) / deg                                # [N, in_dim]
        return self.lin(torch.cat([H, nbr_mean], dim=1))


class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = SAGELayer(in_dim, hidden)
        self.conv2 = SAGELayer(hidden, out_dim)
        self.dropout = dropout

    def forward(self, H, A):
        H = F.relu(self.conv1(H, A))
        H = F.dropout(H, p=self.dropout, training=self.training)
        return self.conv2(H, A)


torch.manual_seed(0)
model = GraphSAGE(in_dim=N, hidden=16, out_dim=2, dropout=0.3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05, weight_decay=5e-4)

for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    logits = model(X, A)
    loss = F.cross_entropy(logits[train_mask], y[train_mask])
    loss.backward()
    optimizer.step()

with torch.no_grad():
    model.eval()
    pred = model(X, A).argmax(dim=1)
    acc = (pred == y).float().mean().item()
print(f'GraphSAGE  full acc {acc:.4f}')


## 5. GAT：图注意力网络

GCN 给每个邻居一样的归一化权重 $\frac{1}{\sqrt{d_i d_j}}$，但实际上有些邻居比另一些更重要——比如社交网络里，知心朋友的信息比泛泛之交更值得听。

**GAT** 用**注意力机制**让模型学习"邻居 $j$ 对节点 $i$ 有多重要"：

$$ \alpha_{ij} = \mathrm{softmax}_{j \in \mathcal{N}(i)}\Big(\mathrm{LeakyReLU}\big(\boldsymbol{a}^\top [\boldsymbol{W}\boldsymbol{h}_i \,\Vert\, \boldsymbol{W}\boldsymbol{h}_j]\big)\Big), $$

$$ \boldsymbol{h}_i' = \sigma\Big(\sum_{j \in \mathcal{N}(i) \cup \{i\}} \alpha_{ij}\ \boldsymbol{W} \boldsymbol{h}_j\Big). $$

实际中通常用**多头注意力**：跑 $K$ 套独立的 $(\boldsymbol{W}, \boldsymbol{a})$ 然后 concat。


In [ ]:
class GATLayer(nn.Module):
    """单头 GAT 层。多头通过堆多个实例 + concat 实现。"""

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.a = nn.Parameter(torch.randn(2 * out_dim) * 0.1)
        self.leaky = nn.LeakyReLU(0.2)

    def forward(self, H, A):
        # H: [N, in_dim]; A: [N, N]; 自环
        N = H.shape[0]
        H_w = self.W(H)                                    # [N, out_dim]

        # 算所有 (i, j) 对的注意力 logits
        # e_ij = LeakyReLU(a^T [Wh_i || Wh_j])
        a_src, a_dst = self.a[:H_w.shape[1]], self.a[H_w.shape[1]:]
        e_src = H_w @ a_src                                # [N]
        e_dst = H_w @ a_dst                                # [N]
        E = e_src.unsqueeze(1) + e_dst.unsqueeze(0)        # [N, N]
        E = self.leaky(E)

        # 只在邻居 + 自环上 softmax，其他位置 mask 成 -inf
        A_self = A + torch.eye(N, device=A.device)
        E = E.masked_fill(A_self == 0, float('-inf'))
        alpha = F.softmax(E, dim=1)                        # 每行做 softmax
        return alpha @ H_w                                 # 加权聚合


class GAT(nn.Module):
    """两层多头 GAT."""

    def __init__(self, in_dim, hidden, out_dim, heads=4, dropout=0.5):
        super().__init__()
        # 第 1 层：多头，concat
        self.layer1_heads = nn.ModuleList([GATLayer(in_dim, hidden) for _ in range(heads)])
        # 第 2 层：单头输出（也可以多头取平均）
        self.layer2 = GATLayer(hidden * heads, out_dim)
        self.dropout = dropout

    def forward(self, H, A):
        H = torch.cat([head(H, A) for head in self.layer1_heads], dim=1)
        H = F.elu(H)
        H = F.dropout(H, p=self.dropout, training=self.training)
        return self.layer2(H, A)


torch.manual_seed(0)
model = GAT(in_dim=N, hidden=8, out_dim=2, heads=4, dropout=0.3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05, weight_decay=5e-4)

for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    logits = model(X, A)
    loss = F.cross_entropy(logits[train_mask], y[train_mask])
    loss.backward()
    optimizer.step()

with torch.no_grad():
    model.eval()
    pred = model(X, A).argmax(dim=1)
    acc = (pred == y).float().mean().item()
print(f'GAT  full acc {acc:.4f}')


## 6. GIN：图同构网络

Xu et al. 从理论上分析了 GNN 的**表达能力**：什么样的 GNN 能区分两个不同的图？他们证明，如果 AGG 是单射函数（injective），GNN 就能达到 1-WL 图同构测试的判别能力。

GIN 用一个简单的设计实现这个：

$$ \boldsymbol{h}_i^{(l+1)} = \mathrm{MLP}^{(l)}\Big((1 + \epsilon) \cdot \boldsymbol{h}_i^{(l)} + \sum_{j \in \mathcal{N}(i)} \boldsymbol{h}_j^{(l)}\Big), $$

要点：

- 用 **sum** 而非 mean/max（sum 是单射，能区分邻居数量）；
- 用 **MLP** 而非单层 linear（MLP 可以拟合任意单射函数）；
- $\epsilon$ 可以是固定常数或可学习参数，区分中心节点的"自身贡献"。


In [ ]:
class GINLayer(nn.Module):
    def __init__(self, in_dim, out_dim, eps=0.0, train_eps=False):
        super().__init__()
        self.eps = nn.Parameter(torch.tensor(eps)) if train_eps else eps
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
        )

    def forward(self, H, A):
        # 邻居 sum + (1+eps) * self
        out = (1 + self.eps) * H + A @ H
        return self.mlp(out)


class GIN(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.5):
        super().__init__()
        self.conv1 = GINLayer(in_dim, hidden)
        self.conv2 = GINLayer(hidden, out_dim)
        self.dropout = dropout

    def forward(self, H, A):
        H = F.relu(self.conv1(H, A))
        H = F.dropout(H, p=self.dropout, training=self.training)
        return self.conv2(H, A)


torch.manual_seed(0)
model = GIN(in_dim=N, hidden=16, out_dim=2, dropout=0.3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05, weight_decay=5e-4)

for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    logits = model(X, A)
    loss = F.cross_entropy(logits[train_mask], y[train_mask])
    loss.backward()
    optimizer.step()

with torch.no_grad():
    model.eval()
    pred = model(X, A).argmax(dim=1)
    acc = (pred == y).float().mean().item()
print(f'GIN  full acc {acc:.4f}')


## 7. 四种 GNN 在 Karate Club 上的对比

我们用同样的训练超参（同一个种子、200 epoch、Adam、lr=0.05、dropout=0.3）跑一遍 GCN/GraphSAGE/GAT/GIN，看在仅有 2 个标签节点的极度半监督设定下，哪种 GNN 最稳健。


In [ ]:
def train_and_eval(model_cls, model_kwargs, use_A_hat=False, n_runs=5):
    """训练多次取平均，避免单次随机性。"""
    accs = []
    for seed in range(n_runs):
        torch.manual_seed(seed)
        model = model_cls(**model_kwargs)
        opt = torch.optim.Adam(model.parameters(), lr=0.05, weight_decay=5e-4)
        adj = A_hat if use_A_hat else A
        for _ in range(200):
            model.train()
            opt.zero_grad()
            logits = model(X, adj)
            loss = F.cross_entropy(logits[train_mask], y[train_mask])
            loss.backward(); opt.step()
        with torch.no_grad():
            model.eval()
            acc = (model(X, adj).argmax(dim=1) == y).float().mean().item()
        accs.append(acc)
    return np.mean(accs), np.std(accs)


results = {}
results['GCN']       = train_and_eval(GCN,       dict(in_dim=N, hidden=16, out_dim=2, dropout=0.3), use_A_hat=True)
results['GraphSAGE'] = train_and_eval(GraphSAGE, dict(in_dim=N, hidden=16, out_dim=2, dropout=0.3))
results['GAT']       = train_and_eval(GAT,       dict(in_dim=N, hidden=8,  out_dim=2, heads=4, dropout=0.3))
results['GIN']       = train_and_eval(GIN,       dict(in_dim=N, hidden=16, out_dim=2, dropout=0.3))

print(f'{"model":<12s}{"mean acc":>10s}{"std":>10s}')
for name, (m, s) in results.items():
    print(f'{name:<12s}{m:>10.4f}{s:>10.4f}')


**典型观察**：在 Karate Club 这种**强同质性**（同社区节点更可能相连）的小图上：

- 4 种 GNN 都能达到 80%+ 的准确率；
- GCN/GIN 通常最稳定（方差小）；
- GAT 在大图上优势更明显（这里 34 节点没有体现）；
- GraphSAGE 的优势是 scale 到大图，这里同样没有体现。

> **常见误区**：把所有 GNN 当成 GCN 的变种。事实上它们的设计动机完全不同：GCN 是 spectral CNN 的近似，GraphSAGE 是面向大图的归纳学习，GAT 是 self-attention 的图变体，GIN 是表达力理论驱动。


## 8. 学到的节点嵌入

把第一层 GCN 输出（隐藏层）用 PCA 降到 2 维，看一下学到的节点表示是否能在空间上自然分开两个派系。


In [ ]:
torch.manual_seed(0)
gcn = GCN(in_dim=N, hidden=16, out_dim=2, dropout=0.0)
opt = torch.optim.Adam(gcn.parameters(), lr=0.05, weight_decay=5e-4)
for _ in range(200):
    gcn.train()
    opt.zero_grad()
    loss = F.cross_entropy(gcn(X, A_hat)[train_mask], y[train_mask])
    loss.backward(); opt.step()

# 取出第一层的输出作为节点嵌入
with torch.no_grad():
    gcn.eval()
    h1 = gcn.conv1(X, A_hat).numpy()

# PCA 到 2 维
from sklearn.decomposition import PCA
emb_2d = PCA(n_components=2).fit_transform(h1)

plt.figure(figsize=(7, 6))
colors = ['tab:blue', 'tab:red']
for k in [0, 1]:
    plt.scatter(emb_2d[y.numpy() == k, 0], emb_2d[y.numpy() == k, 1],
                s=120, alpha=0.7, label=f'class {k}', c=colors[k])
for i in range(N):
    plt.annotate(str(i), (emb_2d[i, 0], emb_2d[i, 1]), fontsize=7, ha='center', va='center')
plt.title('GCN node embeddings (PCA-2D)')
plt.legend(); plt.grid(alpha=0.3); plt.show()


## 9. 玩具案例：分子图回归

GNN 的一个重要应用是**图级任务**（不是节点级）：给一个完整的图，预测一个标量或类别。典型场景是分子性质预测——给一个分子的化学结构图，预测其能量、毒性、溶解度等。

我们造一个超简化的玩具数据集：每个图随机生成 5～10 个节点的小图，每个节点有 3 维特征。**任务**：预测整图的"密度分数" = 边数 / 节点数 × 节点特征之和的某个统计量。

这一节的重点不是结果好坏，而是演示 GNN 在图级任务上的**读出**（readout）操作：把所有节点表示聚合成一个图级表示。


In [ ]:
def random_graph(min_n=5, max_n=10, feat_dim=3, edge_prob=0.4, seed=None):
    """随机生成一个小图（Erdős–Rényi 模型）。"""
    g = torch.Generator()
    if seed is not None:
        g.manual_seed(seed)
    n = torch.randint(min_n, max_n + 1, (1,), generator=g).item()
    Xg = torch.randn(n, feat_dim, generator=g)
    Ag = (torch.rand(n, n, generator=g) < edge_prob).float()
    Ag = (Ag + Ag.t()) / 2
    Ag = (Ag > 0).float()
    Ag.fill_diagonal_(0)
    n_edges = (Ag.sum() / 2).item()
    target = n_edges / n + 0.3 * Xg.sum().item() / n
    return Xg, Ag, torch.tensor(target, dtype=torch.float32)


# 生成数据集
torch.manual_seed(0)
train_data = [random_graph(seed=i) for i in range(200)]
test_data  = [random_graph(seed=1000 + i) for i in range(50)]
print(f'train {len(train_data)} graphs    test {len(test_data)} graphs')


# ============== 图级 GIN：多层消息传递 + 全图 readout ==============
class GraphGIN(nn.Module):
    def __init__(self, in_dim, hidden, n_layers=3, dropout=0.0):
        super().__init__()
        self.convs = nn.ModuleList()
        for i in range(n_layers):
            d_in = in_dim if i == 0 else hidden
            self.convs.append(GINLayer(d_in, hidden))
        # 把每一层 readout 后的 hidden 表示拼起来再回归
        self.lin = nn.Linear(hidden * n_layers, 1)
        self.dropout = dropout

    def forward(self, H, A):
        outs = []
        for conv in self.convs:
            H = F.relu(conv(H, A))
            outs.append(H.sum(dim=0))                    # sum readout: [hidden]
        h_graph = torch.cat(outs, dim=0)                 # [hidden * n_layers]
        return self.lin(F.dropout(h_graph, p=self.dropout, training=self.training)).squeeze()


torch.manual_seed(0)
model = GraphGIN(in_dim=3, hidden=16, n_layers=3, dropout=0.2)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

# 一个简单的"逐图"训练循环（小数据集，直接 for 循环够用）
for epoch in range(50):
    model.train()
    total = 0.0
    for Xg, Ag, target in train_data:
        opt.zero_grad()
        pred = model(Xg, Ag)
        loss = loss_fn(pred, target)
        loss.backward(); opt.step()
        total += loss.item()
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            model.eval()
            test_mse = sum(loss_fn(model(Xg, Ag), t).item() for Xg, Ag, t in test_data) / len(test_data)
        print(f'epoch {epoch+1:2d}  train mse {total/len(train_data):.4f}  test mse {test_mse:.4f}')


**关键观察**：

- 在 `GraphGIN` 里我们用了 **sum readout**（把所有节点的隐藏表示求和），这样图级输出对节点数量是敏感的；
- 我们还**拼接了每一层的 readout**，这是 GIN 原论文推荐的做法——不同深度的层捕捉不同尺度的子结构信息；
- 对真实数据（如 OGB Molecular），通常配合 mini-batch 训练。下面的"小批量训练 tips"给出实战要点。


## 10. 拓展：PyTorch Geometric 等价实现

实际项目中，[PyTorch Geometric](https://pytorch-geometric.readthedocs.io/) 提供了更完整、更高效的 GNN 实现，支持：

- 各种图数据集（Cora、Citeseer、PubMed、OGB、QM9 等）的一键加载；
- 高效的稀疏邻接表表示（`edge_index`）；
- 邻居采样、mini-batch、异构图等；
- 50+ 种 GNN 层（`GCNConv`、`SAGEConv`、`GATConv`、`GINConv` 等）。

如果安装了 PyG，等价的 GCN 写法是：

```python
import torch_geometric as pyg
from torch_geometric.nn import GCNConv

class GCN_PyG(torch.nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, out_dim)
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv2(x, edge_index)
```

PyG 用 `edge_index` `[2, E]` 替代邻接矩阵 `[N, N]`，对稀疏大图（百万节点、亿条边）更省内存。安装：

```bash
pip install torch_geometric
```

> 安装可能需要额外的编译扩展（`torch_scatter`, `torch_sparse`），具体看 PyG 官方文档。


## 小结

本章实现了 4 种主流 GNN：

| 模型 | 关键思路 | 公式核心 |
|---|---|---|
| **GCN** | 对称归一化的图卷积 | $\tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2} H W$ |
| **GraphSAGE** | 邻居采样 + 聚合 | $\sigma(W[h_i\,\Vert\,\mathrm{AGG}(\mathcal{N}(i))])$ |
| **GAT** | 学习邻居权重 | $\sum_j \alpha_{ij} W h_j$ |
| **GIN** | 单射聚合 + MLP | $\mathrm{MLP}((1+\epsilon)h_i + \sum_j h_j)$ |

**实战经验速查**：

- **节点级任务**（Cora、社交网络）通常用 GCN 或 GAT；强同质图用 GCN 足够，弱同质或异质图考虑 GAT/GraphSAGE。
- **图级任务**（分子性质预测）通常用 GIN，因为它表达力理论保证最好；记得加多层 readout。
- **大图**（百万+节点）必须用 GraphSAGE 这类邻居采样方法，或者考虑 Cluster-GCN/GraphSAINT 这类图采样。
- **过度平滑问题**：层数太深时所有节点表示趋同。一般保持 2-3 层；用残差连接、跳跃连接缓解。

下一章我们会进入**大语言模型与智能体**——把注意力机制扩展到序列上的 Transformer 模型，并构建一个简单的智能体。
